In [1]:
# 02_preprocessing.ipynb
# Cross-Domain AI Governance Gap Analysis
# 02. Preprocessing
#
# A single shared preprocessing routine applied consistently to every dataset so that
# credit and insurance are directly comparable:
#   numeric  -> median imputation (fit on train) + standard scaling
#   category -> "missing" fill + one-hot, train/test aligned
#   large sets subsampled to a common row cap; stratified train/test split; splits persisted.

import os
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.25
MAX_ROWS = 40000          # common cap before splitting, to align sample sizes across domains

df_german = pd.read_parquet(os.path.join(PROCESSED_DIR, "german.parquet"))
df_home = pd.read_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval.parquet"))
df_ins = pd.read_parquet(os.path.join(PROCESSED_DIR, "insurance.parquet"))
df_porto = pd.read_parquet(os.path.join(PROCESSED_DIR, "porto_seguro.parquet"))

# Optional stratified subsampling to a common row cap (keeps class balance)
def subsample(df, target_col, max_rows=MAX_ROWS, random_state=RANDOM_STATE):
    if max_rows is None or len(df) <= max_rows:
        return df.copy()
    frac = max_rows / len(df)
    # groupby.sample keeps the grouping column (robust across pandas versions)
    out = df.groupby(target_col, group_keys=False).sample(frac=frac, random_state=random_state)
    return out.reset_index(drop=True)

df_porto_s = subsample(df_porto, target_col="target")
print("Porto Seguro after subsample:", df_porto_s.shape)

# Shared preprocessing helper. Split first, then fit imputation/scaling/encoding on train only.
def preprocess_dataset(df, target_col, sensitive_cols, drop_cols=None,
                       test_size=TEST_SIZE, random_state=RANDOM_STATE):
    df = df.copy()
    drop_cols = drop_cols or []
    y = df[target_col].astype(int)
    sensitive = df[sensitive_cols].copy()
    feature_df = df.drop(columns=[target_col] + sensitive_cols + drop_cols, errors="ignore")
    num_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = feature_df.select_dtypes(exclude=[np.number]).columns.tolist()

    idx_train, idx_test = train_test_split(
        feature_df.index, test_size=test_size, random_state=random_state, stratify=y)
    X_train_raw = feature_df.loc[idx_train].copy()
    X_test_raw = feature_df.loc[idx_test].copy()

    num_medians = X_train_raw[num_cols].median()
    X_train_raw[num_cols] = X_train_raw[num_cols].fillna(num_medians)
    X_test_raw[num_cols] = X_test_raw[num_cols].fillna(num_medians)

    scaler = StandardScaler()
    X_train_num = (pd.DataFrame(scaler.fit_transform(X_train_raw[num_cols]),
                   columns=num_cols, index=X_train_raw.index) if num_cols
                   else pd.DataFrame(index=X_train_raw.index))
    X_test_num = (pd.DataFrame(scaler.transform(X_test_raw[num_cols]),
                  columns=num_cols, index=X_test_raw.index) if num_cols
                  else pd.DataFrame(index=X_test_raw.index))

    if cat_cols:
        X_train_cat = pd.get_dummies(X_train_raw[cat_cols].astype("object").fillna("missing"),
                                     drop_first=False)
        X_test_cat = pd.get_dummies(X_test_raw[cat_cols].astype("object").fillna("missing"),
                                    drop_first=False)
        X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join="left", axis=1, fill_value=0)
        X_test_cat = X_test_cat[X_train_cat.columns]
    else:
        X_train_cat = pd.DataFrame(index=X_train_raw.index)
        X_test_cat = pd.DataFrame(index=X_test_raw.index)

    X_train = pd.concat([X_train_num, X_train_cat], axis=1)
    X_test = pd.concat([X_test_num, X_test_cat], axis=1)

    train = X_train.copy()
    train[target_col] = y.loc[idx_train].values
    for c in sensitive_cols:
        train[c] = sensitive.loc[idx_train, c].values
    test = X_test.copy()
    test[target_col] = y.loc[idx_test].values
    for c in sensitive_cols:
        test[c] = sensitive.loc[idx_test, c].values

    meta = {"target_col": target_col, "sensitive_cols": sensitive_cols,
            "n_features": X_train.shape[1], "n_train": len(train), "n_test": len(test),
            "num_cols": num_cols, "cat_cols": cat_cols}
    return train, test, meta

german_train, german_test, german_meta = preprocess_dataset(
    df_german, "target", ["sex", "age_group"], drop_cols=["personal_status_sex", "domain"])
print("German:", german_train.shape, german_test.shape)

home_alt_target = df_home[["target_60dpd"]].copy()
home_train, home_test, home_meta = preprocess_dataset(
    df_home, "target", ["sex"],
    drop_cols=["ID", "CODE_GENDER", "target_30dpd", "target_60dpd", "domain"])
home_train = home_train.join(home_alt_target, how="left")
home_test = home_test.join(home_alt_target, how="left")
print("Credit Card Approval:", home_train.shape, home_test.shape)

ins_train, ins_test, ins_meta = preprocess_dataset(
    df_ins, "target", ["sex", "age_group", "smoker_flag"],
    drop_cols=["smoker", "charges", "domain"])
print("Medical Cost:", ins_train.shape, ins_test.shape)

proxy_cols = [c for c in df_porto_s.columns if c.startswith("ps_ind") and ("cat" in c or "bin" in c)]
porto_train, porto_test, porto_meta = preprocess_dataset(
    df_porto_s, "target", proxy_cols, drop_cols=["id", "domain"])
print("Porto Seguro:", porto_train.shape, porto_test.shape)

german_train.to_parquet(os.path.join(PROCESSED_DIR, "german_train.parquet"))
german_test.to_parquet(os.path.join(PROCESSED_DIR, "german_test.parquet"))
home_train.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval_train.parquet"))
home_test.to_parquet(os.path.join(PROCESSED_DIR, "credit_card_approval_test.parquet"))
ins_train.to_parquet(os.path.join(PROCESSED_DIR, "insurance_train.parquet"))
ins_test.to_parquet(os.path.join(PROCESSED_DIR, "insurance_test.parquet"))
porto_train.to_parquet(os.path.join(PROCESSED_DIR, "porto_seguro_train.parquet"))
porto_test.to_parquet(os.path.join(PROCESSED_DIR, "porto_seguro_test.parquet"))

metadata = {
    "german": {**german_meta, "domain": "credit"},
    "credit_card_approval": {**home_meta, "domain": "credit"},
    "insurance": {**ins_meta, "domain": "insurance"},
    "porto_seguro": {**porto_meta, "domain": "insurance"},
    "config": {"random_state": RANDOM_STATE, "test_size": TEST_SIZE, "max_rows": MAX_ROWS},
}
with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json"), "w") as f:
    json.dump(metadata, f, indent=2)

prep_summary = pd.DataFrame([
    {"domain": "credit", "dataset": "German Credit", "n_features": german_meta["n_features"],
     "n_train": german_meta["n_train"], "n_test": german_meta["n_test"],
     "sensitive": ", ".join(german_meta["sensitive_cols"])},
    {"domain": "credit", "dataset": "Credit Card Approval", "n_features": home_meta["n_features"],
     "n_train": home_meta["n_train"], "n_test": home_meta["n_test"],
     "sensitive": ", ".join(home_meta["sensitive_cols"])},
    {"domain": "insurance", "dataset": "Medical Cost", "n_features": ins_meta["n_features"],
     "n_train": ins_meta["n_train"], "n_test": ins_meta["n_test"],
     "sensitive": ", ".join(ins_meta["sensitive_cols"])},
    {"domain": "insurance", "dataset": "Porto Seguro", "n_features": porto_meta["n_features"],
     "n_train": porto_meta["n_train"], "n_test": porto_meta["n_test"],
     "sensitive": f"{len(porto_meta['sensitive_cols'])} proxy cols"},
])
print(prep_summary.to_string(index=False))
prep_summary.to_csv(os.path.join(TAB_DIR, "02_preprocess_summary.csv"), index=False)
print("02_preprocessing complete")


Porto Seguro after subsample: (40000, 60)
German: (750, 60) (250, 60)
Credit Card Approval: (27342, 56) (9115, 56)
Medical Cost: (1003, 11) (335, 11)


Porto Seguro: (30000, 58) (10000, 58)
   domain              dataset  n_features  n_train  n_test                   sensitive
   credit        German Credit          57      750     250              sex, age_group
   credit Credit Card Approval          53    27342    9115                         sex
insurance         Medical Cost           7     1003     335 sex, age_group, smoker_flag
insurance         Porto Seguro          43    30000   10000               14 proxy cols
02_preprocessing complete
